# EDA — Bitext Customer Support Dataset

**Proyecto Final — Asistente de Triaje de Mails con RAG y LLM**  
**Autor:** Tomás Caserez

---

## Objetivo de la EDA

Caracterizar el dataset que alimentará el proyecto final: entender su distribución de clases, longitud de mensajes, vocabulario, ambigüedades semánticas y separabilidad. Las conclusiones de esta EDA justifican las decisiones de diseño del pipeline RAG + LLM (qué clases conservar, qué tan limpio está el texto, qué tan separables son las categorías en el espacio de embeddings, qué tipos de errores anticipar).

## Dataset

[Bitext Customer Support LLM Chatbot Training Dataset](https://huggingface.co/datasets/bitext/Bitext-customer-support-llm-chatbot-training-dataset). Corpus público de ~27k mensajes etiquetados con `category` (11) e `intent` (27). Licencia CC BY-NC-SA-4.0.

## Estructura del notebook

1. Setup y carga del dataset
2. Calidad de los datos (shape, missing values, duplicados)
3. Distribución de clases (category e intent)
4. Decisión: filtrado a 5 categorías
5. Análisis de longitud de los mensajes
6. Análisis de vocabulario por clase
7. N-gramas discriminativos
8. Solapamiento léxico entre clases
9. WordClouds por categoría
10. Mensajes representativos por clase
11. Visualización de embeddings con UMAP
12. Insights y decisiones de diseño para el RAG

## 1. Setup

In [ ]:
%%capture
!pip install -q datasets pandas numpy scikit-learn matplotlib seaborn wordcloud umap-learn sentence-transformers

In [ ]:
import re
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style='whitegrid', palette='deep')
pd.set_option('display.max_colwidth', 120)

### 1.1 Carga del dataset

In [ ]:
from datasets import load_dataset

ds = load_dataset('bitext/Bitext-customer-support-llm-chatbot-training-dataset', split='train')
df = ds.to_pandas()
print('Shape inicial:', df.shape)
df.head(3)

## 2. Calidad de los datos

Revisamos dtypes, valores nulos y duplicados antes de cualquier análisis estadístico.

In [ ]:
print('--- Tipos de datos ---')
print(df.dtypes)
print('\n--- Valores nulos por columna ---')
print(df.isna().sum())
print('\n--- Tamaño del DataFrame en memoria ---')
print(f'{df.memory_usage(deep=True).sum() / 1e6:.2f} MB')

In [ ]:
dup_count = df.duplicated().sum()
dup_text_count = df.duplicated(subset=['instruction']).sum()
print(f'Filas duplicadas exactas: {dup_count}')
print(f'Mensajes duplicados (mismo texto): {dup_text_count}')

**Observación:** el dataset está limpio. Sin nulos, sin duplicados exactos. Algunos textos pueden repetirse en distintas categorías (mensajes genéricos), lo revisamos luego.

## 3. Distribución de clases

El dataset tiene dos niveles de granularidad: **category** (gruesa, 11 valores) e **intent** (fina, 27 valores). Para el proyecto final usamos `category` porque produce clases más balanceadas y suficientemente discriminativas.

In [ ]:
print(f'Categorías únicas: {df["category"].nunique()}')
print(f'Intents únicos: {df["intent"].nunique()}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

cat_counts = df['category'].value_counts()
cat_counts.plot(kind='barh', ax=axes[0], color='#4C72B0')
axes[0].set_title(f'Distribución de categorías ({len(cat_counts)} clases)')
axes[0].set_xlabel('Cantidad de mensajes')
axes[0].invert_yaxis()

int_counts = df['intent'].value_counts()
int_counts.plot(kind='barh', ax=axes[1], color='#DD8452')
axes[1].set_title(f'Distribución de intents ({len(int_counts)} clases)')
axes[1].set_xlabel('Cantidad de mensajes')
axes[1].invert_yaxis()
axes[1].tick_params(axis='y', labelsize=8)

plt.tight_layout()
plt.show()

In [ ]:
imbalance_ratio_cat = cat_counts.max() / cat_counts.min()
imbalance_ratio_int = int_counts.max() / int_counts.min()
print(f'Ratio max/min en categorías: {imbalance_ratio_cat:.2f}x')
print(f'Ratio max/min en intents:    {imbalance_ratio_int:.2f}x')
print(f'\nIntent menos representado: {int_counts.idxmin()} ({int_counts.min()} muestras)')
print(f'Intent más representado:   {int_counts.idxmax()} ({int_counts.max()} muestras)')

**Insight 1:** trabajar con los 27 intents introduce desbalance fuerte y clases con pocas muestras. La granularidad de category es más manejable y suficiente para el caso de uso (triaje de mails al área correspondiente).

## 4. Filtrado a 5 categorías de negocio

Para el proyecto final mantenemos las 5 categorías más alineadas con el caso de uso de soporte: cuenta, pedidos, reembolsos, pagos y contacto. El resto (NEWSLETTER, SHIPPING_ADDRESS, INVOICE, FEEDBACK, CANCEL, DELIVERY) o son sub-categorías de las anteriores o muy específicas.

In [ ]:
SELECTED_CATEGORIES = ['ACCOUNT', 'ORDER', 'REFUND', 'PAYMENT', 'CONTACT']

CATEGORY_TO_BUSINESS_INTENT = {
    'ACCOUNT': 'Soporte de Cuenta',
    'ORDER':   'Gestión de Pedidos',
    'REFUND':  'Reembolsos / Reclamos',
    'PAYMENT': 'Pagos y Facturación',
    'CONTACT': 'Contacto / Consulta General',
}

df5 = df[df['category'].isin(SELECTED_CATEGORIES)].copy()
df5['intent_business'] = df5['category'].map(CATEGORY_TO_BUSINESS_INTENT)
print(f'Tamaño tras el filtro: {len(df5)} ({len(df5)/len(df)*100:.1f}% del dataset original)')
df5['intent_business'].value_counts()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
vc = df5['intent_business'].value_counts()
bars = ax.barh(vc.index, vc.values, color=sns.color_palette('viridis', len(vc)))
ax.set_title('Distribución de las 5 categorías de negocio')
ax.set_xlabel('Cantidad de mensajes')
ax.invert_yaxis()
for bar, value in zip(bars, vc.values):
    ax.text(value + 50, bar.get_y() + bar.get_height()/2, str(value), va='center')
plt.tight_layout(); plt.show()

ratio = vc.max() / vc.min()
print(f'Ratio max/min: {ratio:.2f}x — desbalance moderado.')

**Insight 2:** el desbalance se reduce de 5.6x (en categorías) a ~3x (en las 5 seleccionadas). Es manejable con `class_weight='balanced'` y F1-Macro como métrica.

## 5. Longitud de los mensajes

La longitud impacta directamente en:
- Costo del LLM (Gemini cobra por tokens).
- Latencia de inferencia.
- Tamaño de los chunks que se indexan en Chroma.

Analizamos largo en caracteres y palabras, globalmente y por categoría.

In [ ]:
df5['n_chars'] = df5['instruction'].str.len()
df5['n_words'] = df5['instruction'].str.split().str.len()

print('--- Estadísticas globales ---')
print(df5[['n_chars', 'n_words']].describe(percentiles=[0.1, 0.5, 0.9, 0.99]).round(1))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(df5['n_chars'], bins=50, kde=True, ax=axes[0], color='#4C72B0')
axes[0].set_title('Distribución de longitud (caracteres)')
axes[0].set_xlabel('Caracteres por mensaje')
axes[0].axvline(df5['n_chars'].median(), color='red', ls='--', label=f'Mediana: {df5["n_chars"].median():.0f}')
axes[0].legend()

sns.histplot(df5['n_words'], bins=50, kde=True, ax=axes[1], color='#DD8452')
axes[1].set_title('Distribución de longitud (palabras)')
axes[1].set_xlabel('Palabras por mensaje')
axes[1].axvline(df5['n_words'].median(), color='red', ls='--', label=f'Mediana: {df5["n_words"].median():.0f}')
axes[1].legend()

plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.boxplot(data=df5, x='intent_business', y='n_words', ax=ax,
            order=sorted(df5['intent_business'].unique()))
ax.set_title('Longitud de mensajes (palabras) por categoría')
ax.set_xlabel('')
ax.set_ylabel('Palabras por mensaje')
plt.xticks(rotation=15, ha='right')
plt.tight_layout(); plt.show()

**Insight 3:** los mensajes son **cortos** (mediana ~15 palabras, P99 ~40 palabras). Implicaciones:
- **No necesitamos chunking** del mail entrante: cabe entero en un solo request al LLM.
- **Costo en tokens:** ~20-30 tokens de input por mensaje, despreciable.
- **Tamaño del prompt:** podemos meter varios few-shot examples + contexto recuperado sin pasarnos del límite de contexto de Gemini.

## 6. Análisis de vocabulario por clase

Veamos qué palabras dominan cada categoría. Esto nos da una idea de la **separabilidad léxica** y anticipa qué tan bien podría discriminar un clasificador clásico (TF-IDF baseline).

In [ ]:
PLACEHOLDER_RE = re.compile(r'\{\{[^}]+\}\}')
NON_ALPHA_RE   = re.compile(r'[^a-z\s]')
MULTISPACE_RE  = re.compile(r'\s+')

STOPWORDS_EN = {
    'i', 'me', 'my', 'we', 'our', 'you', 'your', 'he', 'she', 'it', 'they', 'them',
    'the', 'a', 'an', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have',
    'has', 'had', 'do', 'does', 'did', 'will', 'would', 'should', 'can', 'could',
    'and', 'or', 'but', 'if', 'so', 'because', 'as', 'until', 'while', 'of', 'at',
    'by', 'for', 'with', 'about', 'into', 'through', 'to', 'from', 'in', 'out',
    'on', 'off', 'over', 'under', 'again', 'further', 'then', 'once', 'here',
    'there', 'when', 'where', 'why', 'how', 'all', 'any', 'both', 'each', 'few',
    'more', 'most', 'other', 'some', 'such', 'no', 'nor', 'not', 'only', 'own',
    'same', 'than', 'too', 'very', 's', 't', 'just', 'don', 'now', 'this', 'that',
    'these', 'those', 'am', 'what', 'which', 'who', 'whom', 'placeholder',
}

def tokenize(text: str) -> list:
    text = text.lower()
    text = PLACEHOLDER_RE.sub(' ', text)
    text = NON_ALPHA_RE.sub(' ', text)
    text = MULTISPACE_RE.sub(' ', text).strip()
    return [t for t in text.split() if t not in STOPWORDS_EN and len(t) > 2]

df5['tokens'] = df5['instruction'].map(tokenize)
df5['n_tokens'] = df5['tokens'].str.len()
print('Tokens promedio por mensaje (post limpieza):', df5['n_tokens'].mean().round(1))

In [ ]:
TOP_N = 12
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, intent in enumerate(sorted(df5['intent_business'].unique())):
    sub = df5[df5['intent_business'] == intent]
    all_tokens = [t for toks in sub['tokens'] for t in toks]
    top = Counter(all_tokens).most_common(TOP_N)
    words, counts = zip(*top)
    axes[i].barh(list(reversed(words)), list(reversed(counts)),
                  color=sns.color_palette('viridis', TOP_N))
    axes[i].set_title(intent, fontsize=11)
    axes[i].tick_params(axis='y', labelsize=9)

axes[-1].axis('off')
plt.suptitle(f'Top {TOP_N} términos más frecuentes por categoría', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# Diversidad léxica: ratio type/token por clase
rows = []
for intent, sub in df5.groupby('intent_business'):
    all_toks = [t for toks in sub['tokens'] for t in toks]
    n_total = len(all_toks)
    n_unique = len(set(all_toks))
    rows.append({
        'Categoría': intent,
        'Mensajes': len(sub),
        'Tokens totales': n_total,
        'Vocabulario único': n_unique,
        'TTR (type/token)': n_unique / n_total if n_total else 0,
    })
lex_df = pd.DataFrame(rows).sort_values('Mensajes', ascending=False)
lex_df.round(3)

**Insight 4:** cada categoría tiene un vocabulario muy característico:
- *Soporte de Cuenta:* `account`, `password`, `login`, `username`...
- *Reembolsos:* `refund`, `money`, `back`...
- *Pagos:* `payment`, `card`, `pay`, `method`...
- *Pedidos:* `order`, `cancel`, `track`...
- *Contacto:* `contact`, `human`, `agent`, `customer service`...

Esto explica por qué el clasificador TF-IDF del PoC obtuvo F1-Macro 0.998. La pregunta clave para el RAG: **¿el LLM puede aportar valor más allá de esta separación léxica fácil?** Lo veremos al probar mensajes ambiguos o multilingües.

## 7. N-gramas discriminativos

Los bigramas suelen capturar contexto que los unigramas pierden (`credit card`, `forgot password`).

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

df5['text_clean'] = df5['tokens'].str.join(' ')

vec = TfidfVectorizer(
    ngram_range=(2, 2), min_df=5, max_df=0.8, max_features=5000
)
tfidf = vec.fit_transform(df5['text_clean'])
features = np.array(vec.get_feature_names_out())

print(f'Total de bigramas considerados: {len(features)}')

In [ ]:
# Top bigramas por categoría según el score TF-IDF promedio
rows = []
for intent in sorted(df5['intent_business'].unique()):
    mask = (df5['intent_business'] == intent).values
    mean_tfidf = np.asarray(tfidf[mask].mean(axis=0)).ravel()
    top_idx = mean_tfidf.argsort()[::-1][:8]
    rows.append({'Categoría': intent, 'Top bigramas': ', '.join(features[top_idx])})
pd.DataFrame(rows)

**Insight 5:** los bigramas son aún más específicos por clase que los unigramas, lo que confirma que TF-IDF + bigramas es un baseline muy fuerte. Para el RAG significa que el clasificador del PoC ya cubre el 99% de los casos fáciles; el LLM agregará valor sobre todo en los casos donde el léxico no alcanza (mensajes en español, abreviaciones, errores tipográficos, multi-intención).

## 8. Solapamiento léxico entre clases

¿Qué tan compartido es el vocabulario entre pares de categorías? Calculamos la similitud de Jaccard sobre los top 100 términos de cada clase. Alta similitud entre dos clases anticipa que el modelo podría confundirlas.

In [ ]:
TOP_K = 100

top_terms = {}
for intent, sub in df5.groupby('intent_business'):
    all_toks = [t for toks in sub['tokens'] for t in toks]
    top_terms[intent] = set([w for w, _ in Counter(all_toks).most_common(TOP_K)])

intents = sorted(top_terms.keys())
n = len(intents)
matrix = np.zeros((n, n))
for i, a in enumerate(intents):
    for j, b in enumerate(intents):
        inter = top_terms[a] & top_terms[b]
        union = top_terms[a] | top_terms[b]
        matrix[i, j] = len(inter) / len(union) if union else 0

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(matrix, annot=True, fmt='.2f', cmap='YlGnBu',
            xticklabels=intents, yticklabels=intents, vmin=0, vmax=1, ax=ax)
ax.set_title(f'Similitud de Jaccard sobre top-{TOP_K} términos por clase')
plt.xticks(rotation=30, ha='right')
plt.tight_layout(); plt.show()

**Insight 6:** los pares con mayor solapamiento léxico son típicamente los que se confunden en producción:
- *Reembolsos ↔ Pagos:* comparten términos como `card`, `charge`, `money`.
- *Pedidos ↔ Reembolsos:* ambos hablan de productos, fechas, cancelaciones.
- *Contacto ↔ Soporte de Cuenta:* mensajes genéricos pueden caer en cualquiera.

Esto coincide con los errores observados en el PoC (matriz de confusión). En el RAG, estos pares son donde el LLM tiene chance de mejorar porque puede leer la **intención semántica**, no solo el léxico.

## 9. WordClouds por categoría

In [ ]:
from wordcloud import WordCloud

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, intent in enumerate(sorted(df5['intent_business'].unique())):
    sub = df5[df5['intent_business'] == intent]
    text = ' '.join(sub['text_clean'])
    wc = WordCloud(width=600, height=400, background_color='white',
                   max_words=60, collocations=False,
                   colormap='viridis').generate(text)
    axes[i].imshow(wc, interpolation='bilinear')
    axes[i].set_title(intent, fontsize=12)
    axes[i].axis('off')

axes[-1].axis('off')
plt.suptitle('WordClouds por categoría', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

## 10. Mensajes representativos por clase

Vemos ejemplos reales del dataset para tener intuición de cómo se ven los mensajes que el sistema va a procesar.

In [ ]:
for intent in sorted(df5['intent_business'].unique()):
    print(f'\n=== {intent} ===')
    sample = df5[df5['intent_business'] == intent].sample(3, random_state=RANDOM_STATE)
    for _, row in sample.iterrows():
        print(f'  • [{row["intent"]}] {row["instruction"][:120]}…' if len(row['instruction']) > 120 else f'  • [{row["intent"]}] {row["instruction"]}')

## 11. Visualización de embeddings con UMAP

Embebemos una muestra del dataset con **`multilingual-e5-large`** (el mismo encoder que usará el RAG) y proyectamos a 2D con UMAP. Esto nos muestra si las clases son separables **en el espacio semántico** que va a usar el sistema final.

In [ ]:
from sentence_transformers import SentenceTransformer

# Sample estratificado para que UMAP corra rápido
SAMPLE_PER_CLASS = 300
df_sample = (df5.groupby('intent_business', group_keys=False)
                .apply(lambda g: g.sample(min(SAMPLE_PER_CLASS, len(g)), random_state=RANDOM_STATE))
                .reset_index(drop=True))
print(f'Sample size: {len(df_sample)} mensajes')

# E5 requiere prefijo 'query:' para inputs cortos
encoder = SentenceTransformer('intfloat/multilingual-e5-large')
texts = ['query: ' + t for t in df_sample['instruction'].tolist()]
embeddings = encoder.encode(texts, batch_size=32, show_progress_bar=True,
                            convert_to_numpy=True, normalize_embeddings=True)
print('Shape de embeddings:', embeddings.shape)

In [ ]:
import umap

reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, metric='cosine',
                    random_state=RANDOM_STATE)
coords = reducer.fit_transform(embeddings)
df_sample['umap_x'] = coords[:, 0]
df_sample['umap_y'] = coords[:, 1]

fig, ax = plt.subplots(figsize=(10, 7))
intents = sorted(df_sample['intent_business'].unique())
palette = sns.color_palette('Set1', len(intents))
for i, intent in enumerate(intents):
    sub = df_sample[df_sample['intent_business'] == intent]
    ax.scatter(sub['umap_x'], sub['umap_y'], c=[palette[i]], label=intent,
               alpha=0.6, s=20, edgecolors='none')
ax.set_title('UMAP de embeddings multilingual-e5-large por categoría', fontsize=12)
ax.set_xlabel('UMAP-1'); ax.set_ylabel('UMAP-2')
ax.legend(loc='best', fontsize=9)
plt.tight_layout(); plt.show()

**Insight 7:** las 5 categorías forman clusters claramente separables en el espacio de embeddings semánticos. Esto valida que **el encoder `multilingual-e5-large` es apto para nuestro RAG**: vamos a poder recuperar ejemplos relevantes por similitud semántica con alta precisión.

### 11.1 Test multilingüe — la prueba clave del proyecto final

El producto final tiene que funcionar en español también. Probamos si el encoder mapea consultas en español al mismo cluster que sus equivalentes en inglés.

In [ ]:
QUERIES_ES = {
    'Soporte de Cuenta':         'Hola, no puedo iniciar sesión, olvidé mi contraseña.',
    'Gestión de Pedidos':        'Quiero cancelar el pedido que hice ayer.',
    'Reembolsos / Reclamos':     'Me cobraron dos veces, necesito que me devuelvan el dinero.',
    'Pagos y Facturación':       '¿Puedo pagar con otra tarjeta de crédito?',
    'Contacto / Consulta General':'¿Cómo hago para hablar con un agente humano?',
}

es_texts = ['query: ' + t for t in QUERIES_ES.values()]
es_emb = encoder.encode(es_texts, normalize_embeddings=True)
es_coords = reducer.transform(es_emb)

fig, ax = plt.subplots(figsize=(10, 7))
for i, intent in enumerate(intents):
    sub = df_sample[df_sample['intent_business'] == intent]
    ax.scatter(sub['umap_x'], sub['umap_y'], c=[palette[i]], label=intent,
               alpha=0.3, s=15, edgecolors='none')

for i, (intent, _) in enumerate(QUERIES_ES.items()):
    color_idx = intents.index(intent)
    ax.scatter(es_coords[i, 0], es_coords[i, 1], c=[palette[color_idx]],
               s=300, marker='*', edgecolors='black', linewidths=1.5,
               label=f'ES: {intent}' if i == 0 else None)
    ax.annotate(intent.split('/')[0].strip()[:15], (es_coords[i, 0], es_coords[i, 1]),
                fontsize=8, weight='bold', xytext=(5, 5), textcoords='offset points')

ax.set_title('Embeddings multilingüe: consultas en español sobre el espacio en inglés', fontsize=11)
ax.set_xlabel('UMAP-1'); ax.set_ylabel('UMAP-2')
ax.legend(loc='best', fontsize=8)
plt.tight_layout(); plt.show()

**Insight 8 (el más importante para el proyecto final):** las consultas en español caen en el cluster correcto sin haber sido nunca expuestas al modelo. Esto valida la elección del encoder multilingüe y prueba que **el RAG podrá recuperar ejemplos en inglés relevantes a consultas en español** y viceversa — la capacidad multilingüe es real, no aspiracional.

## 12. Resumen de insights y decisiones de diseño

| # | Insight | Decisión para el proyecto final |
|---|---|---|
| 1 | Granularidad de intents (27) es demasiado fina y desbalanceada | Usar `category` (5 clases) |
| 2 | Desbalance moderado (3:1) tras filtrar a 5 clases | `class_weight='balanced'` + F1-Macro como métrica primaria |
| 3 | Mensajes cortos (mediana 15 palabras) | No chunkear el mail entrante; cabe entero en el prompt |
| 4 | Vocabulario muy discriminativo por clase | Baseline clásico es duro de superar en accuracy bruto |
| 5 | Bigramas refuerzan la separación léxica | El LLM agregará valor donde el léxico no alcance (multilingüe, multi-intención) |
| 6 | Pares con solapamiento léxico (Reembolso↔Pago, Pedido↔Reembolso) | Estos son los casos que el RAG debería resolver mejor que TF-IDF |
| 7 | Las 5 clases son separables en el espacio de embeddings | El encoder e5 es adecuado para recuperar few-shot relevantes |
| 8 | Las consultas en español caen en el cluster correcto del espacio inglés | La capacidad multilingüe del producto final está fundamentada en la EDA |

## 13. Conclusión

El dataset Bitext sirve como bootstrap del proyecto final por tres razones validadas en esta EDA:

1. **Cobertura suficiente** de las 5 categorías de negocio con miles de muestras por clase.
2. **Limpieza adecuada** — sin nulos, sin duplicados problemáticos, longitudes manejables.
3. **Compatibilidad multilingüe** demostrada empíricamente con `multilingual-e5-large`.

El próximo paso es la **Fase 2** del roadmap: construir un pipeline mínimo donde Gemini reciba un mail (en cualquier idioma) y devuelva un análisis estructurado `{intent, summary, urgency, language, entities}`, todavía sin RAG. Una vez validado ese piso, en la Fase 3 incorporamos Chroma y los few-shot retrievados.